## Install Libraries


In [0]:
%pip install --upgrade databricks-sdk psycopg2-binary
%restart_python


## Lakebase Connection Pool Manager with Automatic Token Refresh

This notebook demonstrates a best practice pattern for managing OAuth tokens with connection pools.

**Key Features:**
- Automatic token refresh before expiry
- Graceful pool transition (no connection failures during refresh)
- Thread-safe connection management
- Background daemon thread for token monitoring


In [0]:
import threading
import time
import uuid
from datetime import datetime, timedelta
from typing import Optional, Any, List, Tuple
import psycopg2
from psycopg2 import pool
from databricks.sdk import WorkspaceClient


class LakebaseConnectionPoolManager:
    """
    Manages a psycopg2 connection pool with automatic OAuth token refresh.
    Recreates the pool before token expiry to ensure continuous availability.
    
    Usage:
        pool_manager = LakebaseConnectionPoolManager(
            workspace_client=w,
            instance_name="ec-lakebase",
            dbname="my_database",
            user="my-spn-client-id",
            refresh_margin_seconds=300  # Refresh 5 min before expiry
        )
        
        # Execute queries
        rows = pool_manager.execute("SELECT * FROM my_table LIMIT 10")
        
        # Or get a connection directly
        conn = pool_manager.getconn()
        try:
            # use connection
        finally:
            pool_manager.putconn(conn)
        
        # Shutdown when done
        pool_manager.shutdown()
    """
    
    def __init__(
        self,
        workspace_client: WorkspaceClient,
        instance_name: str,
        dbname: str,
        user: str,
        minconn: int = 1,
        maxconn: int = 10,
        refresh_margin_seconds: int = 300,  # Refresh 5 min before expiry
        port: str = '5432'
    ):
        """
        Initialize the connection pool manager.
        
        Args:
            workspace_client: Authenticated Databricks WorkspaceClient
            instance_name: Name of the Lakebase database instance
            dbname: Database name to connect to
            user: Service Principal client ID for authentication
            minconn: Minimum number of connections in the pool
            maxconn: Maximum number of connections in the pool
            refresh_margin_seconds: Seconds before token expiry to trigger refresh
            port: Database port (default: 5432)
        """
        self.w = workspace_client
        self.instance_name = instance_name
        self.dbname = dbname
        self.user = user
        self.minconn = minconn
        self.maxconn = maxconn
        self.refresh_margin_seconds = refresh_margin_seconds
        self.port = port
        
        self._pool: Optional[pool.ThreadedConnectionPool] = None
        self._pool_lock = threading.Lock()
        self._token_expiry: Optional[datetime] = None
        self._refresh_thread: Optional[threading.Thread] = None
        self._shutdown = threading.Event()
        
        # Initialize the pool
        self._create_pool()
        
        # Start background refresh thread
        self._start_refresh_thread()
    
    def _get_token_and_expiry(self) -> Tuple[str, datetime]:
        """Generate a new token and return (token, expiry_datetime)."""
        cred = self.w.database.generate_database_credential(
            request_id=str(uuid.uuid4()),
            instance_names=[self.instance_name]
        )
        
        # Parse ISO 8601 expiration time (e.g., '2025-12-10T15:55:06Z')
        expiry_str = cred.expiration_time.replace('Z', '+00:00')
        expiry = datetime.fromisoformat(expiry_str).replace(tzinfo=None)
        
        return cred.token, expiry
    
    def _create_pool(self) -> None:
        """
        Create a new connection pool with fresh credentials.
        Handles graceful transition from old pool to new pool.
        """
        instance = self.w.database.get_database_instance(name=self.instance_name)
        token, expiry = self._get_token_and_expiry()
        
        new_pool = psycopg2.pool.ThreadedConnectionPool(
            minconn=self.minconn,
            maxconn=self.maxconn,
            host=instance.read_write_dns,
            dbname=self.dbname,
            user=self.user,
            password=token,
            sslmode="require",
            port=self.port
        )
        
        with self._pool_lock:
            old_pool = self._pool
            self._pool = new_pool
            self._token_expiry = expiry
            print(f"[{datetime.utcnow().isoformat()}] New pool created. Token expires at: {expiry.isoformat()}")
        
        # Close old pool after a grace period (allow in-flight operations to complete)
        if old_pool:
            threading.Timer(30.0, self._close_pool, args=[old_pool]).start()
    
    def _close_pool(self, pool_to_close: pool.ThreadedConnectionPool) -> None:
        """
        Safely close an old connection pool.
        
        Args:
            pool_to_close: The pool instance to close
        """
        try:
            pool_to_close.closeall()
            print(f"[{datetime.utcnow().isoformat()}] Old pool closed successfully")
        except Exception as e:
            print(f"[{datetime.utcnow().isoformat()}] Error closing old pool: {e}")
    
    def _refresh_loop(self) -> None:
        """
        Background thread that monitors token expiry and refreshes the pool.
        Runs until shutdown is signaled.
        """
        while not self._shutdown.is_set():
            if self._token_expiry:
                time_until_refresh = (
                    self._token_expiry 
                    - datetime.utcnow() 
                    - timedelta(seconds=self.refresh_margin_seconds)
                ).total_seconds()
                
                if time_until_refresh <= 0:
                    print(f"[{datetime.utcnow().isoformat()}] Token refresh triggered...")
                    try:
                        self._create_pool()
                    except Exception as e:
                        print(f"[{datetime.utcnow().isoformat()}] Error refreshing pool: {e}")
                        # Retry in 30 seconds on failure
                        time_until_refresh = 30
                
                # Sleep until next refresh check (check every 60s or when refresh is due)
                sleep_time = min(max(time_until_refresh, 1), 60)
                self._shutdown.wait(timeout=sleep_time)
            else:
                self._shutdown.wait(timeout=60)
    
    def _start_refresh_thread(self) -> None:
        """Start the background token refresh thread."""
        self._refresh_thread = threading.Thread(
            target=self._refresh_loop,
            daemon=True,
            name="LakebaseTokenRefreshThread"
        )
        self._refresh_thread.start()
        print(f"[{datetime.utcnow().isoformat()}] Token refresh thread started")
    
    def getconn(self) -> Any:
        """
        Get a connection from the pool.
        
        Returns:
            A psycopg2 connection object
        """
        with self._pool_lock:
            return self._pool.getconn()
    
    def putconn(self, conn: Any) -> None:
        """
        Return a connection to the pool.
        
        Args:
            conn: The connection to return
        """
        with self._pool_lock:
            if self._pool:
                self._pool.putconn(conn)
    
    def execute(self, query: str, params: Optional[Tuple] = None) -> Optional[List[Tuple]]:
        """
        Execute a query using a connection from the pool.
        
        Args:
            query: SQL query string
            params: Optional tuple of query parameters
        
        Returns:
            List of tuples for SELECT queries, None for other queries
        """
        conn = None
        try:
            conn = self.getconn()
            with conn.cursor() as cur:
                cur.execute(query, params)
                if cur.description:  # SELECT query
                    return cur.fetchall()
                conn.commit()  # For INSERT/UPDATE/DELETE
                return None
        finally:
            if conn:
                self.putconn(conn)
    
    def get_pool_status(self) -> dict:
        """
        Get the current status of the connection pool.
        
        Returns:
            Dictionary with pool status information
        """
        with self._pool_lock:
            time_until_expiry = None
            if self._token_expiry:
                time_until_expiry = (self._token_expiry - datetime.utcnow()).total_seconds()
            
            return {
                "token_expiry": self._token_expiry.isoformat() if self._token_expiry else None,
                "seconds_until_expiry": time_until_expiry,
                "seconds_until_refresh": time_until_expiry - self.refresh_margin_seconds if time_until_expiry else None,
                "refresh_margin_seconds": self.refresh_margin_seconds,
                "pool_active": self._pool is not None,
                "refresh_thread_alive": self._refresh_thread.is_alive() if self._refresh_thread else False
            }
    
    def shutdown(self) -> None:
        """
        Gracefully shutdown the pool manager.
        Stops the refresh thread and closes all connections.
        """
        print(f"[{datetime.utcnow().isoformat()}] Shutting down pool manager...")
        self._shutdown.set()
        
        if self._refresh_thread:
            self._refresh_thread.join(timeout=5)
        
        with self._pool_lock:
            if self._pool:
                self._pool.closeall()
                self._pool = None
        
        print(f"[{datetime.utcnow().isoformat()}] Pool manager shutdown complete")
    
    def __enter__(self):
        """Context manager entry."""
        return self
    
    def __exit__(self, exc_type, exc_val, exc_tb):
        """Context manager exit - ensures cleanup."""
        self.shutdown()
        return False


## Initialize the Pool Manager


In [0]:
# Configuration
DATABRICKS_HOST = "https://adb-984752964297111.11.azuredatabricks.net/"
AZURE_CLIENT_ID = "449bbe09-5ee9-4d0f-a985-5d8e1b84e708"
AZURE_TENANT_ID = "9f37a392-f0ae-4280-9796-f1864a10effc"
INSTANCE_NAME = "ec-lakebase"
DATABASE_NAME = "uc_demos_elmer_cecilio"

# Get the SPN secret from Databricks secrets
spn_secret = dbutils.secrets.get(scope="ec_secrets", key="spn_secret")

# Create the Workspace Client
w = WorkspaceClient(
    host=DATABRICKS_HOST,
    azure_client_id=AZURE_CLIENT_ID,
    azure_tenant_id=AZURE_TENANT_ID,
    azure_client_secret=spn_secret
)

# Create the Pool Manager with automatic token refresh
# Token will be refreshed 5 minutes (300 seconds) before expiry
pool_manager = LakebaseConnectionPoolManager(
    workspace_client=w,
    instance_name=INSTANCE_NAME,
    dbname=DATABASE_NAME,
    user=AZURE_CLIENT_ID,
    minconn=1,
    maxconn=10,
    refresh_margin_seconds=300  # Refresh 5 minutes before expiry
)

print("\nPool Manager initialized successfully!")


## Check Pool Status


In [0]:
# Check the current status of the pool
status = pool_manager.get_pool_status()

print("Pool Status:")
print("-" * 50)
print(f"Token Expiry:           {status['token_expiry']}")
print(f"Seconds Until Expiry:   {status['seconds_until_expiry']:.2f}" if status['seconds_until_expiry'] else "N/A")
print(f"Seconds Until Refresh:  {status['seconds_until_refresh']:.2f}" if status['seconds_until_refresh'] else "N/A")
print(f"Refresh Margin:         {status['refresh_margin_seconds']} seconds")
print(f"Pool Active:            {status['pool_active']}")
print(f"Refresh Thread Alive:   {status['refresh_thread_alive']}")


## Execute Queries


In [0]:
# Example 1: Using the execute() helper method
rows = pool_manager.execute("""
    SELECT * FROM demo.products_lakebase LIMIT 10
""")

print("Query Results:")
print("-" * 80)
for row in rows:
    print(row)


In [0]:
# Example 2: Using getconn/putconn for more control
conn = pool_manager.getconn()
try:
    with conn.cursor() as cur:
        cur.execute("SELECT version()")
        version = cur.fetchone()
        print(f"PostgreSQL Version: {version[0]}")
finally:
    pool_manager.putconn(conn)


In [0]:
# Example 3: Parameterized query
category = 'Electronics'
rows = pool_manager.execute(
    "SELECT * FROM demo.products_lakebase WHERE category = %s LIMIT 5",
    (category,)
)

print(f"Products in '{category}' category:")
print("-" * 80)
for row in rows:
    print(row)


## Simulate Long-Running Process

This cell demonstrates that the connection pool will automatically refresh tokens in the background while your application continues to work.


In [0]:
# Simulate a long-running process that periodically queries the database
# The token refresh happens automatically in the background

def simulate_workload(pool_manager, duration_seconds=60, query_interval=10):
    """
    Simulate a workload that queries the database periodically.
    Token refresh happens automatically in the background.
    
    Args:
        pool_manager: The LakebaseConnectionPoolManager instance
        duration_seconds: How long to run the simulation
        query_interval: Seconds between queries
    """
    start_time = time.time()
    query_count = 0
    
    print(f"Starting workload simulation for {duration_seconds} seconds...")
    print(f"Querying every {query_interval} seconds")
    print("=" * 60)
    
    while time.time() - start_time < duration_seconds:
        try:
            # Execute a simple query
            result = pool_manager.execute("SELECT COUNT(*) FROM demo.products_lakebase")
            query_count += 1
            
            # Get pool status
            status = pool_manager.get_pool_status()
            
            print(f"[{datetime.utcnow().isoformat()}] Query #{query_count}: COUNT = {result[0][0]} | "
                  f"Token expires in {status['seconds_until_expiry']:.0f}s")
            
        except Exception as e:
            print(f"[{datetime.utcnow().isoformat()}] Query error: {e}")
        
        time.sleep(query_interval)
    
    print("=" * 60)
    print(f"Workload simulation complete. Executed {query_count} queries.")

# Run for 60 seconds, querying every 10 seconds
# In production, this would run continuously and the token would refresh automatically
simulate_workload(pool_manager, duration_seconds=60, query_interval=10)


## Manual Token Refresh (Optional)

If you need to manually trigger a token refresh (e.g., after detecting an auth error), you can call `_create_pool()` directly.


In [0]:
# Force a manual token refresh
print("Before refresh:")
print(pool_manager.get_pool_status())

print("\nTriggering manual refresh...")
pool_manager._create_pool()

print("\nAfter refresh:")
print(pool_manager.get_pool_status())


## Cleanup

Always shut down the pool manager when you're done to clean up resources.


In [0]:
# Gracefully shutdown the pool manager
pool_manager.shutdown()

# Verify shutdown
print("\nFinal status:")
print(pool_manager.get_pool_status())


## Alternative: Using Context Manager

The pool manager also supports context manager syntax for automatic cleanup.


In [0]:
# Re-initialize for context manager example
spn_secret = dbutils.secrets.get(scope="ec_secrets", key="spn_secret")

w = WorkspaceClient(
    host=DATABRICKS_HOST,
    azure_client_id=AZURE_CLIENT_ID,
    azure_tenant_id=AZURE_TENANT_ID,
    azure_client_secret=spn_secret
)

# Using context manager for automatic cleanup
with LakebaseConnectionPoolManager(
    workspace_client=w,
    instance_name=INSTANCE_NAME,
    dbname=DATABASE_NAME,
    user=AZURE_CLIENT_ID,
    refresh_margin_seconds=300
) as pm:
    # Execute queries
    rows = pm.execute("SELECT * FROM demo.products_lakebase LIMIT 5")
    print("Query Results:")
    for row in rows:
        print(row)
    
    print("\nPool Status:")
    print(pm.get_pool_status())

# Pool is automatically shutdown when exiting the context
print("\nContext exited - pool was automatically cleaned up")
